<a href="https://colab.research.google.com/github/vu21eece0100463/traffic-light-detection/blob/main/Traffic_Light_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q opencv-python pillow

In [ ]:
import cv2
import numpy as np
import json
import time
from datetime import datetime

# Open laptop camera
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot open laptop camera")

# Traffic head ROI
# Adjust based on camera position
ROI = (200, 100, 350, 400)

previous_status = None


def detect_lamp(roi, lamp):

    if roi.size == 0:
        return False

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    if lamp == "RED":

        mask1 = cv2.inRange(
            hsv,
            np.array([0, 120, 160]),
            np.array([10, 255, 255])
        )

        mask2 = cv2.inRange(
            hsv,
            np.array([170, 120, 160]),
            np.array([180, 255, 255])
        )

        mask = mask1 | mask2

    elif lamp == "AMBER":

        mask = cv2.inRange(
            hsv,
            np.array([11, 100, 160]),
            np.array([35, 255, 255])
        )

    elif lamp == "GREEN":

        mask = cv2.inRange(
            hsv,
            np.array([36, 70, 120]),
            np.array([90, 255, 255])
        )

    # Calculate percentage of illuminated pixels
    active_pixels = cv2.countNonZero(mask)

    total_pixels = roi.shape[0] * roi.shape[1]

    percentage = (active_pixels / total_pixels) * 100

    # Threshold for ON/OFF
    return percentage > 5


while True:

    ret, frame = cap.read()

    if not ret:
        break

    x1, y1, x2, y2 = ROI

    traffic_head = frame[y1:y2, x1:x2]

    if traffic_head.size == 0:
        print("Invalid traffic head ROI")
        break

    height = traffic_head.shape[0]

    section = height // 3

    # Divide into 3 lamps
    red_roi = traffic_head[0:section, :]

    amber_roi = traffic_head[
        section:section * 2, :
    ]

    green_roi = traffic_head[
        section * 2:height, :
    ]

    # Detect individual lamp status
    red_on = detect_lamp(red_roi, "RED")

    amber_on = detect_lamp(amber_roi, "AMBER")

    green_on = detect_lamp(green_roi, "GREEN")

    status = {
        "traffic_head_id": 1,
        "red": "ON" if red_on else "OFF",
        "amber": "ON" if amber_on else "OFF",
        "green": "ON" if green_on else "OFF"
    }

    # Print JSON only when state changes
    if status != previous_status:

        output = {
            "timestamp": datetime.now().isoformat(),
            **status
        }

        print(json.dumps(output, indent=4))

        previous_status = status.copy()

    # Draw traffic head rectangle
    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (255, 255, 255),
        2
    )

    lamp_names = ["RED", "AMBER", "GREEN"]

    lamp_states = [red_on, amber_on, green_on]

    lamp_colors = [
        (0, 0, 255),
        (0, 255, 255),
        (0, 255, 0)
    ]

    for i in range(3):

        y_start = y1 + i * section

        y_end = (
            y1 + (i + 1) * section
            if i < 2 else y2
        )

        color = (
            lamp_colors[i]
            if lamp_states[i]
            else (100, 100, 100)
        )

        cv2.rectangle(
            frame,
            (x1, y_start),
            (x2, y_end),
            color,
            2
        )

        cv2.putText(
            frame,
            f"{lamp_names[i]}: "
            f"{'ON' if lamp_states[i] else 'OFF'}",
            (x2 + 10, y_start + 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    cv2.imshow(
        "Traffic Head ON/OFF Detection",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Show the image which was just taken.
  import matplotlib.pyplot as plt
  img = cv2.imread(filename)
  plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

In [ ]:

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
import json
import time
from datetime import datetime

# ==========================================================
# 1. JAVASCRIPT WEBCAM STREAM
# ==========================================================

def video_stream_script():
    js = Javascript("""
    var video;
    var div;
    var stream;
    var canvas;

    async function initCamera() {
        div = document.createElement('div');
        div.id = "webcam-container";

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = 640;
        video.height = 480;
        video.autoplay = true;

        stream = await navigator.mediaDevices.getUserMedia({
            video: {
                width: 640,
                height: 480
            }
        });

        video.srcObject = stream;
        await video.play();

        canvas = document.createElement('canvas');
        canvas.width = 640;
        canvas.height = 480;

        document.body.appendChild(div);
        div.appendChild(video);

        const stopBtn = document.createElement('button');
        stopBtn.textContent = 'Stop Stream';

        stopBtn.onclick = () => {
            stream.getTracks().forEach(track => track.stop());
            div.remove();
        };

        div.appendChild(stopBtn);
    }

    async function captureFrame() {
        if (!stream ||
            stream.getVideoTracks().length === 0 ||
            stream.getVideoTracks()[0].readyState === 'ended') {
            return null;
        }

        const ctx = canvas.getContext('2d');

        ctx.drawImage(video, 0, 0, 640, 480);

        return canvas.toDataURL('image/jpeg', 0.8);
    }
    """)

    display(js)


# ==========================================================
# 2. TRAFFIC LIGHT COLOR DETECTION
# ==========================================================

def detect_lamp(roi, lamp):

    if roi.size == 0:
        return False

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    if lamp == "RED":

        mask1 = cv2.inRange(
            hsv,
            np.array([0, 120, 160]),
            np.array([10, 255, 255])
        )

        mask2 = cv2.inRange(
            hsv,
            np.array([170, 120, 160]),
            np.array([180, 255, 255])
        )

        mask = mask1 | mask2

    elif lamp == "AMBER":

        mask = cv2.inRange(
            hsv,
            np.array([11, 100, 160]),
            np.array([35, 255, 255])
        )

    elif lamp == "GREEN":

        mask = cv2.inRange(
            hsv,
            np.array([36, 70, 120]),
            np.array([90, 255, 255])
        )

    active_pixels = cv2.countNonZero(mask)

    total_pixels = roi.shape[0] * roi.shape[1]

    if total_pixels == 0:
        return False

    percentage = (active_pixels / total_pixels) * 100

    return percentage > 5


# ==========================================================
# 3. CONFIGURATION
# ==========================================================

# Adjust according to the traffic signal position.
# Camera dimensions: 640 x 480

ROI = (200, 50, 400, 430)

traffic_head_id = 1

# How frequently to print JSON while state remains unchanged
JSON_INTERVAL = 1.0  # seconds


# ==========================================================
# 4. TRAFFIC LIGHT DURATION VARIABLES
# ==========================================================

previous_status = None

active_status = "UNKNOWN"

state_start_time = None

last_json_time = 0

# Use monotonic clock for elapsed duration
start_clock = time.monotonic()


# ==========================================================
# 5. START CAMERA
# ==========================================================

video_stream_script()

eval_js('initCamera()')

print("Traffic light monitoring started.")
print("Click Stop Stream to stop camera.")


# ==========================================================
# 6. CONTINUOUS TRAFFIC LIGHT MONITORING
# ==========================================================

try:

    while True:

        frame_data = eval_js('captureFrame()')

        if not frame_data:
            print("Camera stopped.")
            break

        # Decode webcam frame
        binary = b64decode(frame_data.split(',')[1])

        img_array = np.frombuffer(
            binary,
            dtype=np.uint8
        )

        frame = cv2.imdecode(
            img_array,
            cv2.IMREAD_COLOR
        )

        if frame is None:
            continue

        # Extract traffic light region
        x1, y1, x2, y2 = ROI

        traffic_head = frame[y1:y2, x1:x2]

        if traffic_head.size == 0:
            continue

        height = traffic_head.shape[0]

        section = height // 3

        # Divide into 3 lamp regions
        red_roi = traffic_head[0:section, :]

        amber_roi = traffic_head[
            section:section * 2, :
        ]

        green_roi = traffic_head[
            section * 2:height, :
        ]

        # Detect lamp colors
        red_on = detect_lamp(red_roi, "RED")

        amber_on = detect_lamp(amber_roi, "AMBER")

        green_on = detect_lamp(green_roi, "GREEN")


        # ==================================================
        # 7. DETERMINE ACTIVE TRAFFIC LIGHT
        # ==================================================

        if red_on and not amber_on and not green_on:

            current_status = "RED"

        elif amber_on and not red_on and not green_on:

            current_status = "AMBER"

        elif green_on and not red_on and not amber_on:

            current_status = "GREEN"

        else:

            # No lamp or conflicting detections
            current_status = "UNKNOWN"


        # ==================================================
        # 8. CALCULATE STATE DURATION
        # ==================================================

        current_time = time.monotonic()

        # New state detected
        if current_status != active_status:

            active_status = current_status

            state_start_time = current_time

            print(
                f"Traffic light changed to: {active_status}"
            )

        # Calculate elapsed time for current state
        if state_start_time is not None:

            duration_seconds = (
                current_time - state_start_time
            )

        else:

            duration_seconds = 0


        # ==================================================
        # 9. CREATE STATUS JSON
        # ==================================================

        status = {

            "traffic_head_id": traffic_head_id,

            "red": "ON" if red_on else "OFF",

            "amber": "ON" if amber_on else "OFF",

            "green": "ON" if green_on else "OFF",

            "active_status": active_status,

            "duration_seconds": round(
                duration_seconds, 2
            ),

            "duration_minutes": round(
                duration_seconds / 60, 2
            )

        }


        # ==================================================
        # 10. PRINT JSON EVERY SECOND OR ON STATE CHANGE
        # ==================================================

        state_changed = (
            status["active_status"] !=
            (
                previous_status["active_status"]
                if previous_status else None
            )
        )

        if (
            state_changed or
            current_time - last_json_time >= JSON_INTERVAL
        ):

            output = {

                "timestamp": datetime.now().isoformat(),

                **status

            }

            print(
                json.dumps(
                    output,
                    indent=4
                )
            )

            previous_status = status.copy()

            last_json_time = current_time


except KeyboardInterrupt:

    print("Monitoring interrupted.")

except Exception as e:

    print("Monitoring stopped:", str(e))


<IPython.core.display.Javascript object>

Streaming output truncated to the last 5000 lines.
{
    "timestamp": "2026-11-02T10:09:33.594191",
    "traffic_head_id": 1,
    "red": "OFF",
    "amber": "OFF",
    "green": "OFF",
    "active_status": "UNKNOWN",
    "duration_seconds": 64.94,
    "duration_minutes": 1.08
}
{
    "timestamp": "2026-11-02T10:09:34.717641",
    "traffic_head_id": 1,
    "red": "OFF",
    "amber": "OFF",
    "green": "OFF",
    "active_status": "UNKNOWN",
    "duration_seconds": 66.06,
    "duration_minutes": 1.1
}
{
    "timestamp": "2026-11-02T10:09:35.791491",
    "traffic_head_id": 1,
    "red": "OFF",
    "amber": "OFF",
    "green": "OFF",
    "active_status": "UNKNOWN",
    "duration_seconds": 67.13,
    "duration_minutes": 1.12
}
{
    "timestamp": "2026-11-02T10:09:36.890139",
    "traffic_head_id": 1,
    "red": "OFF",
    "amber": "OFF",
    "green": "OFF",
    "active_status": "UNKNOWN",
    "duration_seconds": 68.23,
    "duration_minutes": 1.14
}
{
    "timestamp": "2026-11-02T10:09:37.96